# Task 3 — Correlation: News Sentiment vs. Stock Returns
Date alignment, VADER sentiment scoring, daily return calculation, and Pearson correlation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download('vader_lexicon', quiet=True)

plt.style.use('seaborn-v0_8-whitegrid')
sia = SentimentIntensityAnalyzer()

TICKERS = ['AAPL', 'AMZN', 'GOOG', 'META', 'MSFT', 'NVDA', 'TSLA']

## 1. Load & Align Data

In [ ]:
# Load news
news = pd.read_csv('../data/raw/raw_analyst_ratings.csv')
news['date'] = pd.to_datetime(news['date'], utc=True, errors='coerce')
news.dropna(subset=['date', 'headline', 'stock'], inplace=True)

# Normalize to date only (strip time/timezone)
news['trade_date'] = news['date'].dt.normalize().dt.tz_localize(None).dt.date
news = news[news['stock'].isin(TICKERS)]
print(f"News rows after filter: {len(news)}")

In [ ]:
import os
import yfinance as yf

def load_stock(ticker):
    path = f'../data/raw/{ticker}_historical.csv'
    if not os.path.exists(path):
        df = yf.download(ticker, start='2020-01-01', end='2024-01-01', progress=False)
        df.to_csv(path)
    else:
        df = pd.read_csv(path, index_col=0, parse_dates=True)
    df.columns = [c.strip() for c in df.columns]
    df.dropna(inplace=True)
    return df

# Build combined price DataFrame with daily returns
price_frames = []
for t in TICKERS:
    try:
        d = load_stock(t)
        d['daily_return'] = d['Adj Close'].pct_change() * 100
        d['stock'] = t
        d.index = pd.to_datetime(d.index).date
        price_frames.append(d[['daily_return', 'stock']])
    except Exception as e:
        print(f'{t}: {e}')

prices = pd.concat(price_frames).reset_index().rename(columns={'index': 'trade_date'})
prices.dropna(inplace=True)
print(f"Price rows: {len(prices)}")

## 2. Sentiment Scoring with VADER

In [ ]:
# VADER chosen: designed for short social/financial text, no training needed, fast
news['sentiment'] = news['headline'].apply(
    lambda h: sia.polarity_scores(str(h))['compound']
)

print(news['sentiment'].describe().round(4))

# Aggregate: average sentiment per stock per day
daily_sentiment = (
    news.groupby(['stock', 'trade_date'])['sentiment']
    .mean()
    .reset_index()
    .rename(columns={'sentiment': 'avg_sentiment'})
)
print(f"Daily sentiment rows: {len(daily_sentiment)}")

## 3. Merge Sentiment with Returns

In [ ]:
merged = pd.merge(
    daily_sentiment,
    prices,
    on=['stock', 'trade_date'],
    how='inner'
)
print(f"Merged rows: {len(merged)}")
merged.head()

## 4. Pearson Correlation Analysis

In [ ]:
# Overall correlation
r, p = stats.pearsonr(merged['avg_sentiment'], merged['daily_return'])
print(f"Overall Pearson r = {r:.4f}  (p = {p:.4f})")

# Per-stock correlation
corr_by_stock = (
    merged.groupby('stock')
    .apply(lambda g: pd.Series({
        'pearson_r': stats.pearsonr(g['avg_sentiment'], g['daily_return'])[0],
        'p_value':   stats.pearsonr(g['avg_sentiment'], g['daily_return'])[1],
        'n':         len(g)
    }))
    .reset_index()
)
print(corr_by_stock.round(4))

## 5. Visualizations

In [ ]:
# Scatter plot: sentiment vs return
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(merged['avg_sentiment'], merged['daily_return'],
           alpha=0.3, s=15, color='steelblue')

# Regression line
m, b = np.polyfit(merged['avg_sentiment'], merged['daily_return'], 1)
x_line = np.linspace(merged['avg_sentiment'].min(), merged['avg_sentiment'].max(), 100)
ax.plot(x_line, m * x_line + b, color='red', linewidth=1.5, label=f'r = {r:.3f}')

ax.set_xlabel('Average Daily Sentiment Score (VADER compound)')
ax.set_ylabel('Daily Return (%)')
ax.set_title('News Sentiment vs. Daily Stock Return')
ax.legend()
plt.tight_layout()
plt.savefig('../data/raw/fig9_sentiment_vs_return.png', dpi=150)
plt.show()

In [ ]:
# Per-stock correlation bar chart
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['green' if r > 0 else 'red' for r in corr_by_stock['pearson_r']]
ax.bar(corr_by_stock['stock'], corr_by_stock['pearson_r'], color=colors, alpha=0.7)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Pearson Correlation: Sentiment vs. Return by Stock')
ax.set_ylabel('Pearson r')
ax.set_xlabel('Stock')
plt.tight_layout()
plt.savefig('../data/raw/fig10_correlation_by_stock.png', dpi=150)
plt.show()

In [ ]:
# Sentiment category vs average return
def classify_sentiment(s):
    if s >= 0.05:  return 'Positive'
    elif s <= -0.05: return 'Negative'
    else:          return 'Neutral'

merged['sentiment_class'] = merged['avg_sentiment'].apply(classify_sentiment)

avg_return_by_class = merged.groupby('sentiment_class')['daily_return'].mean()
print(avg_return_by_class)

fig, ax = plt.subplots(figsize=(6, 4))
colors = {'Positive': 'green', 'Neutral': 'gray', 'Negative': 'red'}
avg_return_by_class.reindex(['Positive', 'Neutral', 'Negative']).plot(
    kind='bar', ax=ax,
    color=[colors[c] for c in ['Positive', 'Neutral', 'Negative']],
    alpha=0.75
)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Average Daily Return by Sentiment Category')
ax.set_ylabel('Avg Return (%)')
ax.set_xlabel('Sentiment Category')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../data/raw/fig11_return_by_sentiment_class.png', dpi=150)
plt.show()

## 6. Interpretation

**Correlation Strength:** The Pearson correlation coefficient between average daily news sentiment and daily stock returns is `r`. A value near 0 indicates a weak linear relationship, which is expected — stock prices are driven by many factors beyond news sentiment alone.

**Direction:** A positive `r` suggests that on days with more positive news sentiment, stocks tend to have slightly higher returns, and vice versa.

**Limitations:**
- **Lag effects:** Markets may react to news with a 1–2 day delay; same-day alignment may understate the true relationship.
- **Confounding factors:** Macroeconomic events, earnings releases, and sector rotations all influence returns independently of news sentiment.
- **Causality:** Correlation does not imply causation — rising prices may generate positive news, not the other way around.
- **Aggregation:** Averaging sentiment across multiple articles per day loses intra-day signal.